In [ ]:
# imports
import torch
from models.bert import ContinuityBERT
from train import parse_args
import create_knowledge_graph as kg_utils
from data import utils

In [2]:
# Load the right model that we want to analyze
def create_bert_model(config):
    encoder_type = config["encoder_type"]
    use_kg = "kg" in config["model_type"]
    model = ContinuityBERT(
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        n_gnn_layers=config["n_gnn_layers"],
        hidden_dim=config["hidden_dim"],
        input_dim=utils.SENTENCE_ENCODER_DIM[encoder_type],
        use_kg=use_kg,
        kg_node_dim=kg_utils.KG_NODE_DIM,
        kg_edge_dim=kg_utils.KG_EDGE_DIM,
        dropout=config["dropout"],
        gnn_type=config["gnn_type"],
    )
    return model

config_bert_kg_gat = {
    "train_ratio": 0.5,
    "batch_size": 64,
    "n_continuity_errors": 1, #[1, 2
    "n_heads": 8,
    "n_layers": 3,
    "n_gnn_layers": 2,
    "hidden_dim": 20,
    "dropout": 0.2,
    "n_epochs": 100,
    "n_runs": 5,
    "lr": 1e-5,
    "pr_threshold": 0.3,
    "encoder_type": "all-MiniLM-L6-v2",
    "gnn_type": "gatv2", #["gatv2", "gcn"],
    "model_type": "bert_kg"
}

model = create_bert_model(config_bert_kg_gat)

initialized continuityBERT with 628261 parameters.


In [3]:
# Load saved weights into 
MODEL_WEIGHTS_PATH = "./results/bert_kg_gat/bert_kg_gat-1_error-params.pkl"
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH))

<All keys matched successfully>

In [4]:
# Load example data
import pickle as pkl
input_data = "./data/dataset/encoded/test/test_1_error.pkl"
with open(input_data, "rb") as f:
    dataset, _ = pkl.load(f)

xs, ys = [], []
kgs = []
docs = []
for i, (x, y, kg, doc) in enumerate(dataset):
    xs.append(x)
    ys.append(y)
    kgs.append(kg)
    docs.append(doc)

In [108]:
# 5, 8, 24
example_datapoint = 5
x = xs[example_datapoint]
y = ys[example_datapoint]
kg = kgs[example_datapoint]

# datapoint 5 == 1_error/test/synthetic_kaggle_2000_776_continuity7.txt
# datapoint 8 == .../synthetic_kaggle_2254_779_continuity4.txt
import os
osl = os.listdir
ospj = os.path.join
orig_docs_path = "data/dataset/1_error/test/"
def find_original_doc(kg, y, kgidx=0):
    node_labels = kg["node_labels"]
    if len(node_labels) <= kgidx: return [(None, None)]
    ds = [x for x in osl(orig_docs_path) if x.endswith(".txt")]
    matches = []
    for d in ds:
        with open(ospj(orig_docs_path, d)) as f:
            lines = f.readlines()
        txtmatch = node_labels[kgidx]
        max_idx = torch.argmax(y).item()
        #if d == "synthetic_kaggle_2701_902_continuity6.txt":
        #    print(max_idx, lines[0])
        #    print(str(max_idx) in lines[0])
        #    print(txtmatch in " ".join(lines))
        #    print(txtmatch)
        if not (f"[{max_idx}]" in lines[0] and txtmatch in " ".join(lines)):
            continue
        matches.append((d, lines))
    if not matches:
        print(f"WARNING: no match found for kgidx={kgidx}!")
        return find_original_doc(kg, y, kgidx=kgidx+1)
        matches = [(None, None)]
    print(f"Success! {len(matches)} matching doc(s) found at kgidx={kgidx}")

    # filter matches if more than 1
    kgi = kgidx
    while len(matches) > 1:
        new_matches = []
        kgi += 1
        print(f"DEBUG: filtering with kgi={kgi}")
        if kgi >= len(node_labels):
            print(f"WARNING: unfilterable down to 1 orig data doc! {len(matches)} matches remaining")
            break
        txtmatch = node_labels[kgi]
        for d, lines in matches:
            if not txtmatch in " ".join(lines):
                continue
            new_matches.append((d, lines))
        if new_matches:
            matches = new_matches

    return matches#[0]
        
        

print(f"data_{example_datapoint} # sentences: {len(x)}")
orig_data_file, orig_data_lines = find_original_doc(kg, y)[0]
orig_data_lines = [l.strip() for l in orig_data_lines]
print(f"Using data file: {orig_data_file}")

data_5 # sentences: 126
Success! 1 matching doc(s) found at kgidx=0
Using data file: synthetic_kaggle_2000_776_continuity7.txt


In [109]:
y_hat = model.forward(x.reshape([1, len(x), -1]), [kg])[0]

# Get solutions
def top_idxs_ordered_desc(input: torch.Tensor):
    return [y[1] for y in sorted([(x, i) for i, x in enumerate(input)])[::-1]]

max_idx = torch.argmax(y)
max_idx_hat = torch.argmax(y_hat)
print("max index y:", max_idx)
ordered_desc_idxs = top_idxs_ordered_desc(y_hat)
print("ordered desc indices y:", ordered_desc_idxs)
print("len desc indices y:", len(ordered_desc_idxs))

matched = max_idx == max_idx_hat
print(f"Is correct?: {matched}")

max index y: tensor(6)
ordered desc indices y: [6, 8, 12, 17, 42, 5, 73, 86, 13, 102, 107, 54, 25, 53, 38, 100, 39, 51, 16, 3, 124, 99, 9, 37, 101, 64, 117, 23, 106, 125, 0, 91, 45, 105, 89, 108, 111, 30, 50, 2, 72, 122, 103, 7, 69, 84, 71, 74, 121, 52, 49, 93, 97, 55, 44, 90, 56, 36, 41, 95, 4, 63, 47, 59, 119, 96, 34, 76, 61, 24, 65, 87, 109, 120, 114, 92, 123, 11, 85, 48, 94, 67, 98, 35, 115, 33, 29, 19, 31, 21, 58, 81, 20, 82, 14, 10, 77, 1, 118, 32, 28, 68, 83, 110, 70, 104, 27, 78, 60, 40, 43, 88, 75, 66, 18, 113, 57, 112, 46, 22, 80, 15, 26, 79, 116, 62]
len desc indices y: 126
Is correct?: True


In [110]:
print("Correct sentence:")
print(f"s{ordered_desc_idxs[0]+1:3d}: {orig_data_lines[ordered_desc_idxs[0]+1]}")
print()
print("Next best sentences:")
N = 5
i = 1
found = 0
while found < N:
    sentence_idx = ordered_desc_idxs[i]+1 # +1 because the first sentence is labels
    if sentence_idx >= len(orig_data_lines):
        i += 1
        continue
    print(f"s{sentence_idx+1:3d}: {orig_data_lines[sentence_idx]}")
    i += 1
    found += 1

Correct sentence:
s  7: You not sir are obviously not a good person, otherwise I wouldn't be here.

Next best sentences:
s 10: "What are you talking about I'm an upstand" "Shut up, I don't need to hear your bullshit.
s 14: So don't bore me with your upstanding citizen, deacon of the church, great father crap.
s 19: Replaying the nights he cheated on his wife while she was getting cancer treatment.
s 44: Take the money I've left you and leave.
s  7: Why I'm here is.
